In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/src/task4_enriched_profit

In [0]:
from pyspark.sql import functions as F

In [0]:
# mimic profit_by_year table 
enriched_orders = spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
profit_df = build_profit_by_year(enriched_orders)
profit_df = dedupe(profit_df)
delta_writer(profit_df, "az_ci_de_dbs.ecom_dev.test_profit_by_year")

In [0]:
# Test Source table 

def test_profit_by_year_table_exists():
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.test_profit_by_year"), "test_profit_by_year table does not exist"


def test_profit_by_year_table_not_empty():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.test_profit_by_year")
    assert df.count() > 0, "profit_by_year table is empty"


def test_profit_by_year_table_expected_columns():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.test_profit_by_year")
    expected = ["Year", "product_category", "product_sub_category", "customer", "total_profit"]
    assert sorted(df.columns) == sorted(expected), f"Expected {sorted(expected)}, got {sorted(df.columns)}"

In [0]:
# Test Profit by Year

def _query_profit_by_year():
    return spark.sql("""
        SELECT year, ROUND(SUM(total_profit), 2) AS profit 
        FROM az_ci_de_dbs.ecom_dev.test_profit_by_year 
        GROUP BY year 
        ORDER BY year
    """)


def test_profit_by_year_not_empty():
    result = _query_profit_by_year()
    assert result.count() > 0, "Profit by Year query returned empty"


def test_profit_by_year_columns():
    result = _query_profit_by_year()
    assert result.columns == ["year", "profit"], f"Expected ['year', 'profit'], got {result.columns}"


def test_profit_by_year_no_null_year():
    result = _query_profit_by_year()
    nulls = result.filter(F.col("year").isNull()).count()
    assert nulls == 0, f"Found {nulls} null years"


def test_profit_by_year_no_null_profit():
    result = _query_profit_by_year()
    nulls = result.filter(F.col("profit").isNull()).count()
    assert nulls == 0, f"Found {nulls} null profit values"


def test_profit_by_year_profit_rounded():
    result = _query_profit_by_year()
    violations = result.filter(F.col("profit") != F.round(F.col("profit"), 2)).count()
    assert violations == 0, f"Found {violations} profit values not rounded to 2 places"


def test_profit_by_year_unique_years():
    result = _query_profit_by_year()
    total = result.count()
    distinct = result.select("year").distinct().count()
    assert total == distinct, f"Duplicate years found: {total} rows, {distinct} distinct"


def test_profit_by_year_valid_year_range():
    result = _query_profit_by_year()
    min_yr = result.agg(F.min("year")).collect()[0][0]
    max_yr = result.agg(F.max("year")).collect()[0][0]
    assert min_yr >= 2000, f"Year too low: {min_yr}"
    assert max_yr <= 2030, f"Year too high: {max_yr}"

In [0]:
# Test Profit by Year + Product Category

def _query_profit_by_year_category():
    return spark.sql("""
        SELECT year, product_category, ROUND(SUM(total_profit), 2) AS profit 
        FROM az_ci_de_dbs.ecom_dev.test_profit_by_year 
        GROUP BY year, product_category 
        ORDER BY year, product_category
    """)


def test_profit_by_year_category_not_empty():
    result = _query_profit_by_year_category()
    assert result.count() > 0, "Profit by Year+Category query returned empty"


def test_profit_by_year_category_columns():
    result = _query_profit_by_year_category()
    assert result.columns == ["year", "product_category", "profit"], f"Got {result.columns}"


def test_profit_by_year_category_no_nulls():
    result = _query_profit_by_year_category()
    for col_name in ["year", "product_category", "profit"]:
        nulls = result.filter(F.col(col_name).isNull()).count()
        assert nulls == 0, f"Found {nulls} nulls in {col_name}"


def test_profit_by_year_category_profit_rounded():
    result = _query_profit_by_year_category()
    violations = result.filter(F.col("profit") != F.round(F.col("profit"), 2)).count()
    assert violations == 0, f"Found {violations} profit values not rounded to 2 places"


def test_profit_by_year_category_unique_groups():
    result = _query_profit_by_year_category()
    total = result.count()
    distinct = result.select("year", "product_category").distinct().count()
    assert total == distinct, f"Duplicate groups: {total} rows, {distinct} distinct"


def test_profit_by_year_category_more_rows_than_year_only():
    """Year+Category should have more rows than Year alone (more granular)."""
    year_count = _query_profit_by_year().count()
    cat_count = _query_profit_by_year_category().count()
    assert cat_count >= year_count, f"Year+Category ({cat_count}) should have >= rows than Year ({year_count})"


def test_profit_by_year_category_sums_match_year_totals():
    """Sum of profit per year across categories should match Profit by Year totals."""
    year_totals = _query_profit_by_year().collect()
    cat_totals = _query_profit_by_year_category()
    for row in year_totals:
        yr, expected_profit = row["year"], row["profit"]
        cat_sum = cat_totals.filter(F.col("year") == yr).agg(F.round(F.sum("profit"), 2)).collect()[0][0]
        assert abs(cat_sum - expected_profit) < 0.01, f"Year {yr}: category sum ({cat_sum}) != year total ({expected_profit})"

In [0]:
# Test Profit by Customer

def _query_profit_by_customer():
    return spark.sql("""
        SELECT customer, ROUND(SUM(total_profit), 2) AS profit 
        FROM az_ci_de_dbs.ecom_dev.test_profit_by_year 
        GROUP BY customer 
        ORDER BY customer
    """)


def test_profit_by_customer_not_empty():
    result = _query_profit_by_customer()
    assert result.count() > 0, "Profit by Customer query returned empty"


def test_profit_by_customer_columns():
    result = _query_profit_by_customer()
    assert result.columns == ["customer", "profit"], f"Got {result.columns}"


def test_profit_by_customer_no_null_customer():
    result = _query_profit_by_customer()
    nulls = result.filter(F.col("customer").isNull()).count()
    assert nulls == 0, f"Found {nulls} null customer values"


def test_profit_by_customer_no_blank_customer():
    result = _query_profit_by_customer()
    blanks = result.filter(F.trim(F.col("customer")) == "").count()
    assert blanks == 0, f"Found {blanks} blank customer values"


def test_profit_by_customer_no_null_profit():
    result = _query_profit_by_customer()
    nulls = result.filter(F.col("profit").isNull()).count()
    assert nulls == 0, f"Found {nulls} null profit values"


def test_profit_by_customer_profit_rounded():
    result = _query_profit_by_customer()
    violations = result.filter(F.col("profit") != F.round(F.col("profit"), 2)).count()
    assert violations == 0, f"Found {violations} profit values not rounded to 2 places"


def test_profit_by_customer_unique_customers():
    result = _query_profit_by_customer()
    total = result.count()
    distinct = result.select("customer").distinct().count()
    assert total == distinct, f"Duplicate customers: {total} rows, {distinct} distinct"


def test_profit_by_customer_total_matches_overall():
    """Sum across all customers should match total across all years."""
    cust_total = _query_profit_by_customer().agg(F.round(F.sum("profit"), 2)).collect()[0][0]
    year_total = _query_profit_by_year().agg(F.round(F.sum("profit"), 2)).collect()[0][0]
    assert abs(cust_total - year_total) < 0.01, f"Customer total ({cust_total}) != Year total ({year_total})"

In [0]:
# Test Profit by Customer + Year

def _query_profit_by_customer_year():
    return spark.sql("""
        SELECT year, customer, ROUND(SUM(total_profit), 2) AS profit 
        FROM az_ci_de_dbs.ecom_dev.test_profit_by_year 
        GROUP BY year, customer 
        ORDER BY year, customer
    """)


def test_profit_by_customer_year_not_empty():
    result = _query_profit_by_customer_year()
    assert result.count() > 0, "Profit by Customer+Year query returned empty"


def test_profit_by_customer_year_columns():
    result = _query_profit_by_customer_year()
    assert result.columns == ["year", "customer", "profit"], f"Got {result.columns}"


def test_profit_by_customer_year_no_nulls():
    result = _query_profit_by_customer_year()
    for col_name in ["year", "customer", "profit"]:
        nulls = result.filter(F.col(col_name).isNull()).count()
        assert nulls == 0, f"Found {nulls} nulls in {col_name}"


def test_profit_by_customer_year_profit_rounded():
    result = _query_profit_by_customer_year()
    violations = result.filter(F.col("profit") != F.round(F.col("profit"), 2)).count()
    assert violations == 0, f"Found {violations} profit values not rounded to 2 places"


def test_profit_by_customer_year_unique_groups():
    result = _query_profit_by_customer_year()
    total = result.count()
    distinct = result.select("year", "customer").distinct().count()
    assert total == distinct, f"Duplicate groups: {total} rows, {distinct} distinct"


def test_profit_by_customer_year_more_granular():
    """Customer+Year should have >= rows than Customer alone."""
    cust_count = _query_profit_by_customer().count()
    cust_year_count = _query_profit_by_customer_year().count()
    assert cust_year_count >= cust_count, f"Customer+Year ({cust_year_count}) should have >= rows than Customer ({cust_count})"


def test_profit_by_customer_year_sums_match_year_totals():
    """Sum of customer profit per year should match Profit by Year totals."""
    year_totals = _query_profit_by_year().collect()
    cust_year = _query_profit_by_customer_year()
    for row in year_totals:
        yr, expected_profit = row["year"], row["profit"]
        cust_sum = cust_year.filter(F.col("year") == yr).agg(F.round(F.sum("profit"), 2)).collect()[0][0]
        assert abs(cust_sum - expected_profit) < 0.01, f"Year {yr}: customer sum ({cust_sum}) != year total ({expected_profit})"

In [0]:
# Run all tests
def main_profit_metrics_test():
    test_functions = [
        # source table
        test_profit_by_year_table_exists,
        test_profit_by_year_table_not_empty,
        test_profit_by_year_table_expected_columns,
        # profit by year
        test_profit_by_year_not_empty,
        test_profit_by_year_columns,
        test_profit_by_year_no_null_year,
        test_profit_by_year_no_null_profit,
        test_profit_by_year_profit_rounded,
        test_profit_by_year_unique_years,
        test_profit_by_year_valid_year_range,
        # profit by year + category
        test_profit_by_year_category_not_empty,
        test_profit_by_year_category_columns,
        test_profit_by_year_category_no_nulls,
        test_profit_by_year_category_profit_rounded,
        test_profit_by_year_category_unique_groups,
        test_profit_by_year_category_more_rows_than_year_only,
        test_profit_by_year_category_sums_match_year_totals,
        # profit by customer
        test_profit_by_customer_not_empty,
        test_profit_by_customer_columns,
        test_profit_by_customer_no_null_customer,
        test_profit_by_customer_no_blank_customer,
        test_profit_by_customer_no_null_profit,
        test_profit_by_customer_profit_rounded,
        test_profit_by_customer_unique_customers,
        test_profit_by_customer_total_matches_overall,
        # profit by customer + year
        test_profit_by_customer_year_not_empty,
        test_profit_by_customer_year_columns,
        test_profit_by_customer_year_no_nulls,
        test_profit_by_customer_year_profit_rounded,
        test_profit_by_customer_year_unique_groups,
        test_profit_by_customer_year_more_granular,
        test_profit_by_customer_year_sums_match_year_totals,
    ]

    passed, failed = 0, 0
    for fn in test_functions:
        try:
            fn()
            passed += 1
            print(f"  [PASSED] {fn.__name__}")
        except Exception as e:
            failed += 1
            print(f"  [FAILED] {fn.__name__}: {e}")

    print(f"Results: {passed} passed, {failed} failed, {passed + failed} total")


    # Cleanup
    spark.sql("DROP TABLE IF EXISTS az_ci_de_dbs.ecom_dev.test_profit_by_year")
    print("[CLEANUP] Dropped az_ci_de_dbs.ecom_dev.test_profit_by_year")